# Sentinel-2 - optical band combinations

Sentinel-2 is a multispectral imager: 13 bands at 10/20/60 m. A quicklook picks
**three** of them and maps them to red, green and blue. Which three you choose
decides what the image shows.

| Product | Bands (R, G, B) | Shows |
|---|---|---|
| `true_color_vegetation` | B4, B3, B2 | natural colour, as the eye would see it |
| `false_color_vegetation` | B8A, B4, B3 | near-infrared → vegetation glows red |
| `false_color_glacier` | B12, B8A, B3 | SWIR → separates snow, ice and cloud |

Because the bands have different native resolutions, they are stacked into a
VRT at the finest resolution present and warped together, so the three arrive
co-registered on one grid.

The library reaches bands through GDAL's `SENTINEL2` subdataset abstraction, so
**L1C and L2A both work** despite their different granule layouts.

In [ ]:
# --- papermill parameters -------------------------------------------------
# Leave everything as-is to run against the committed test fixtures (what CI
# does). Point any of these at real data for the full pipeline.
DATA_ROOT = ""      # mounted NBS archive, e.g. "/data/nbsArchive"
SAFE_PATH = ""      # explicit .SAFE / .zip product
IDENTIFIER = ""     # catalogue UUID, resolved via pysent.archive
ENDPOINT = None     # None -> NBS_SENTINEL_CSW_ENDPOINT / https://nbs.csw.met.no
OUTPUT_DIR = "_output"
PLATFORM = "S2"

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import rasterio

import pysent
import nbtools

print("pysent", pysent.__version__, "from", Path(pysent.__file__).parent)

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Find a product

In [ ]:
product = nbtools.resolve_input(
    PLATFORM,
    safe_path=SAFE_PATH or None,
    identifier=IDENTIFIER or None,
    data_root=DATA_ROOT or None,
    endpoint=ENDPOINT,
)
print(nbtools.describe(product))

## 3. The shipped band combinations

In [ ]:
from pysent.s2 import S2_DEFAULT_PRODUCTS, S2_SUPPORTED_BANDS, normalize_sentinel_s2_product_map

for name, bands in S2_DEFAULT_PRODUCTS.items():
    print(f"{name:24s} R={bands[0]:4s} G={bands[1]:4s} B={bands[2]:4s}")

print("\nvalidated against:", S2_SUPPORTED_BANDS)

# Defining your own combination - validated, so a typo fails loudly rather than
# producing a silently wrong image.
custom = normalize_sentinel_s2_product_map({"agriculture": ["B11", "B8A", "B2"]})
print("custom product:", custom)
try:
    normalize_sentinel_s2_product_map({"typo": ["B4", "B3", "B99"]})
except ValueError as exc:
    print("rejected as expected:", exc)

## 4. Read and stretch

In [ ]:
from pysent.s2 import S2_STRETCH_PERCENTILES, stretch_sentinel_s2_rgb

with rasterio.open(product.path) as src:
    rgb = src.read([1, 2, 3]).astype(np.float32)
    nodata = src.nodata

stretched, stats = stretch_sentinel_s2_rgb(rgb, nodata=nodata, percentiles=S2_STRETCH_PERCENTILES)

print(f"{'band':>6} | {'raw min':>8} | {'raw max':>8} | {'p_low':>8} | {'p_high':>8}")
print("-" * 52)
for i, (lo, hi) in enumerate(zip(stats["p_low"], stats["p_high"])):
    band = rgb[i][rgb[i] > 0]
    print(f"{('RGB'[i]):>6} | {band.min():8.0f} | {band.max():8.0f} | {lo:8.0f} | {hi:8.0f}")

## 5. Min/max versus percentile - a real trade-off

The active ingestion path stretches each band by its **min/max**. It is fast and
needs no sorting, but it is decided by the two most extreme pixels in the scene:
a single sunlit cloud top compresses everything else into the bottom of the
range.

A **percentile** clip ignores the tails and is almost always the better picture.
`_write_stretched_sentinel_s2_rgb_percentile` implements it and is ready to be
wired in - the comparison below is exactly the evidence needed to make that call.

In [ ]:
valid = rgb[0] > 0

def _minmax(data):
    out = np.zeros_like(data, dtype=np.uint8)
    for i in range(data.shape[0]):
        band = data[i]
        lo, hi = band[band > 0].min(), band.max()
        out[i] = np.clip((band - lo) / max(hi - lo, 1) * 255, 0, 255).astype(np.uint8)
    return out

minmax = _minmax(rgb)
print(f"{'method':>12} | {'range used':>10} | {'clipped':>8}")
print("-" * 36)
for label, image in (("min/max", minmax), ("percentile", stretched)):
    values = image[0][valid]
    p2, p98 = np.percentile(values, [2, 98])
    clipped = ((values == 0) | (values == 255)).mean()
    print(f"{label:>12} | {(p98 - p2) / 255:9.1%} | {clipped:7.1%}")
print("\nHigher 'range used' with low 'clipped' is better: the image fills the")
print("8-bit range without destroying detail at either end.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
raw_display = np.transpose(rgb, (1, 2, 0))
axes[0].imshow((raw_display - raw_display.min()) / max(np.ptp(raw_display), 1))
axes[0].set_title("raw (unstretched)")
axes[1].imshow(np.transpose(minmax, (1, 2, 0)))
axes[1].set_title("min/max stretch (active in production)")
axes[2].imshow(np.transpose(stretched, (1, 2, 0)))
axes[2].set_title(f"percentile {S2_STRETCH_PERCENTILES}")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

## 6. The full pipeline\n`process_sentinel_s2_safe` does warp → stretch → write for every requested product, fanning out across products with a thread pool.

In [ ]:
if product.can_warp:
    from pysent.s2 import process_sentinel_s2_safe

    name = "true_color_vegetation"
    results = process_sentinel_s2_safe(
        input_dataset=str(product.path),
        output_dir=OUTPUT_DIR,
        product_bands={name: S2_DEFAULT_PRODUCTS[name]},
        output_names={name: f"quicklook_{name}.tif"},
        processing_options={"histogram_stretch": True, "compression": "DEFLATE"},
    )
    for entry in results:
        print(entry)
else:
    print("Fixture mode - no SAFE to warp. Set DATA_ROOT/SAFE_PATH to run this cell.")

## 7. Tuning notes\n\n- **Switching the active writer to percentile** is the main outstanding quality decision; the comparison above is the evidence for it.\n- **The stretch is not free at full resolution** - `np.percentile` over 3 × 10980² is seconds of work. Estimating the percentiles from a decimated read costs almost no accuracy.\n- **Add band combinations** by extending `S2_DEFAULT_PRODUCTS`, or pass `product_bands` per call.\n\nMeasure any of these with [04_benchmarks.ipynb](04_benchmarks.ipynb).